# IA como Ventaja Competitiva

Data flywheel, análisis de moat y narrativa para inversores.

In [ ]:
import anthropic
import json
from pathlib import Path
from datetime import datetime

client = anthropic.Anthropic()
print("Taller de ventaja competitiva listo")

## 1. Data Flywheel: capturar feedback para mejorar el producto

In [ ]:
class DataFlywheel:
    def __init__(self, directorio: str = "flywheel_demo"):
        self.dir = Path(directorio)
        self.dir.mkdir(exist_ok=True)
        self.datos: list = []

    def registrar(self, uid: str, inp: str, out: str,
                  feedback: str = None, score: int = None) -> str:
        item = {
            "id": f"{uid}_{int(datetime.now().timestamp())}",
            "ts": datetime.now().isoformat(),
            "uid": uid, "input": inp, "output": out,
            "feedback": feedback, "score": score
        }
        self.datos.append(item)
        ruta = self.dir / f"{datetime.now().strftime('%Y%m')}.jsonl"
        with open(ruta, "a", encoding="utf-8") as f:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
        return item["id"]

    def metricas(self) -> dict:
        total = len(self.datos)
        if total == 0:
            return {}
        con_fb = [d for d in self.datos if d.get("feedback")]
        positivos = sum(1 for d in con_fb if d["feedback"] == "positivo")
        con_score = [d["score"] for d in self.datos if d.get("score")]
        return {
            "total_interacciones": total,
            "con_feedback_pct": round(len(con_fb) / total * 100, 1),
            "csat_pct": round(positivos / max(len(con_fb), 1) * 100, 1),
            "score_medio": round(sum(con_score) / len(con_score), 2) if con_score else None,
            "interacciones_negativas": sum(1 for d in con_fb if d["feedback"] == "negativo")
        }

    def analizar_fallos(self) -> dict:
        negativas = [d for d in self.datos if d.get("feedback") == "negativo" or
                     (d.get("score") and d["score"] <= 2)]
        if not negativas:
            return {"mensaje": "Sin suficientes datos negativos"}

        ejemplos = "\n".join(
            f"Input: {d['input'][:80]} | Output: {d['output'][:80]}"
            for d in negativas[:5]
        )
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=400,
            messages=[{
                "role": "user",
                "content": f"""Analiza estas {len(negativas)} interacciones con feedback negativo.
Identifica patrones y sugiere mejoras al system prompt.

{ejemplos}

JSON: {{"patrones": ["patrón"], "causa_raiz": "...",
        "mejoras_prompt": ["mejora"], "casos_prueba": ["caso"]}}"""
            }]
        )
        txt = resp.content[0].text
        if "```" in txt:
            txt = txt.split("```")[1].lstrip("json\n")
        try:
            return json.loads(txt)
        except json.JSONDecodeError:
            return {"analisis": txt}


flywheel = DataFlywheel()

INTERACCIONES = [
    ("u01", "Analiza cláusula de limitación total de responsabilidad",
     "ALTO RIESGO: La cláusula elimina toda responsabilidad del proveedor...", "positivo", 5),
    ("u02", "¿Qué significa NDA?",
     "Un NDA es un acuerdo de confidencialidad...", "positivo", 4),
    ("u03", "Revisa mi contrato completo de trabajo",
     "No puedo procesar contratos completos sin texto específico.", "negativo", 2),
    ("u04", "¿Cuándo caduca la garantía?",
     "No tengo acceso a los plazos de tu contrato.", "negativo", 1),
    ("u05", "Cláusula de propiedad intelectual: todo el código es del proveedor",
     "ALTO RIESGO: El cliente pierde todos los derechos sobre el código...", "positivo", 5),
    ("u06", "Renovación automática anual sin previo aviso",
     "ALTO RIESGO: La renovación automática sin notificación puede generar...", "positivo", 5),
    ("u07", "Explica el contrato en general",
     "Necesito que me indiques una cláusula específica para analizar.", "negativo", 2),
]

for uid, inp, out, fb, sc in INTERACCIONES:
    flywheel.registrar(uid, inp, out, fb, sc)

print("MÉTRICAS DEL FLYWHEEL")
print("=" * 40)
for k, v in flywheel.metricas().items():
    print(f"  {k}: {v}")

In [ ]:
print("\nANÁLISIS DE FALLOS (para mejorar el prompt)")
print("=" * 50)
fallos = flywheel.analizar_fallos()
for k, v in fallos.items():
    if isinstance(v, list):
        print(f"\n{k}:")
        for item in v:
            print(f"  • {item}")
    else:
        print(f"\n{k}: {v}")

## 2. Evaluador de moat

In [ ]:
DIMENSIONES_MOAT = {
    "datos_propietarios": "¿Tienes datos exclusivos que tu competencia no puede replicar?",
    "mejora_con_uso": "¿El producto mejora automáticamente con más usuarios (data flywheel)?",
    "coste_de_cambio": "¿Sería costoso o difícil para el cliente migrar a un competidor?",
    "integracion_profunda": "¿Estás integrado en el workflow diario del cliente?",
    "efecto_de_red": "¿El producto es más valioso cuantos más usuarios tiene?",
    "marca_de_nicho": "¿Eres reconocido como referente en tu segmento específico?"
}

def evaluar_moat(respuestas: dict) -> dict:
    puntuacion = sum(1 for v in respuestas.values() if v)
    nivel = "fuerte" if puntuacion >= 4 else "medio" if puntuacion >= 2 else "débil"
    debilidades = [DIMENSIONES_MOAT[k] for k, v in respuestas.items() if not v]

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": f"""Una startup de IA tiene moat {nivel} ({puntuacion}/6 puntos).
Áreas sin moat: {chr(10).join(f'- {d}' for d in debilidades)}

¿Qué 3 acciones concretas debe ejecutar en los próximos 90 días para fortalecer su posición?
Sé específico, práctico y prioriza por impacto."""
        }]
    )
    return {
        "puntuacion": f"{puntuacion}/6",
        "nivel_moat": nivel,
        "fortalezas": [DIMENSIONES_MOAT[k] for k, v in respuestas.items() if v],
        "debilidades": debilidades,
        "acciones_90_dias": resp.content[0].text
    }


# Evaluación de la startup de análisis de contratos
mi_startup = {
    "datos_propietarios": True,    # 4.200 contratos con feedback de abogados
    "mejora_con_uso": True,        # flywheel activo
    "coste_de_cambio": False,      # fácil de reemplazar aún
    "integracion_profunda": False, # no integrado en ERP/workflow todavía
    "efecto_de_red": False,        # no hay efecto de red claro
    "marca_de_nicho": True         # conocidos en PYMEs del sector legal
}

eval_moat = evaluar_moat(mi_startup)

print(f"EVALUACIÓN DE MOAT: {eval_moat['puntuacion']} ({eval_moat['nivel_moat'].upper()})")
print("=" * 50)
print("\n✅ Fortalezas:")
for f in eval_moat["fortalezas"]:
    print(f"   • {f}")
print("\n❌ Debilidades:")
for d in eval_moat["debilidades"]:
    print(f"   • {d}")
print("\n🎯 Acciones próximos 90 días:")
print(eval_moat["acciones_90_dias"])

## 3. Narrativa para inversores

In [ ]:
def generar_narrativa_inversores(producto: str, metricas: dict, diferenciadores: list) -> str:
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system="""Eres un asesor experto en fundraising para startups de IA.
Conoces qué buscan los inversores Series A: moat, data flywheel, unit economics saludables.""",
        messages=[{
            "role": "user",
            "content": f"""Redacta la sección 'Ventaja Competitiva' para el pitch deck:

PRODUCTO: {producto}

MÉTRICAS:
{json.dumps(metricas, ensure_ascii=False, indent=2)}

DIFERENCIADORES:
{chr(10).join(f'- {d}' for d in diferenciadores)}

Responde a estas preguntas implícitas del inversor:
1. ¿Por qué no puede Google/Microsoft hacer esto mañana?
2. ¿Cómo mejora el producto con cada usuario?
3. ¿Cuál es el coste de cambio real?
4. ¿Qué pasa con los márgenes cuando pasas de 500 a 5.000 clientes?

4 párrafos concisos. Lenguaje de negocios. Con datos. Máximo 350 palabras."""
        }]
    )
    return resp.content[0].text


narrativa = generar_narrativa_inversores(
    producto="SaaS de análisis automático de contratos para PYMEs españolas. El usuario sube el contrato, recibe en 30 segundos análisis de riesgos con recomendaciones.",
    metricas={
        "usuarios_activos": 280, "contratos_analizados": 4200,
        "nps": 62, "churn_mensual_pct": 4.2,
        "mrr_eur": 8400, "cac_eur": 95, "ltv_eur": 420,
        "ltv_cac_ratio": 4.4, "payback_meses": 5
    },
    diferenciadores=[
        "Base de datos propietaria: 4.200 contratos con feedback de abogados certificados",
        "Sistema de aprendizaje continuo: cada análisis mejora el modelo",
        "Integración nativa con los 3 principales ERP legales españoles",
        "Conocimiento profundo de jurisprudencia española y GDPR"
    ]
)

print("SECCIÓN 'VENTAJA COMPETITIVA' PARA EL PITCH DECK")
print("=" * 60)
print(narrativa)

## 4. Análisis competitivo automático

In [ ]:
def analizar_competidores(mi_producto: str, competidores: list) -> dict:
    competidores_txt = "\n".join(
        f"- {c['nombre']}: {c['descripcion']} | Precio: {c.get('precio','N/A')} | Debilidad: {c.get('debilidad','N/A')}"
        for c in competidores
    )
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        messages=[{
            "role": "user",
            "content": f"""Analiza la posición competitiva:

MI PRODUCTO: {mi_producto}

COMPETIDORES:
{competidores_txt}

JSON:
{{
  "ventajas_reales": ["con evidencia concreta"],
  "vulnerabilidades": ["dónde eres más débil"],
  "oportunidades": ["hueco de mercado no cubierto"],
  "riesgo_principal": "el mayor riesgo competitivo",
  "estrategia_90_dias": "qué hacer ahora mismo en 2 frases",
  "kpis_a_crecer": ["métrica para construir moat"]
}}"""
        }]
    )
    txt = resp.content[0].text
    if "```" in txt:
        txt = txt.split("```")[1].lstrip("json\n")
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return {"analisis": txt}


analisis = analizar_competidores(
    "Análisis automático de contratos IA para PYMEs, especializado en derecho español",
    [
        {"nombre": "LegalZoom EU", "descripcion": "Plantillas de contratos online",
         "precio": "29€/mes", "debilidad": "No analiza contratos de terceros"},
        {"nombre": "Ironclad", "descripcion": "CLM enterprise para grandes empresas",
         "precio": ">500€/mes", "debilidad": "Muy caro y complejo para PYMEs"},
        {"nombre": "ChatGPT Plus", "descripcion": "IA generalista",
         "precio": "20€/mes", "debilidad": "Sin especialización en derecho español"}
    ]
)

print("ANÁLISIS COMPETITIVO")
print("=" * 60)
for k, v in analisis.items():
    if isinstance(v, list):
        print(f"\n{k}:")
        for item in v:
            print(f"  • {item}")
    else:
        print(f"\n{k}: {v}")